In [1]:
%%writefile app.py
import sys
import io
# 1. Filtre absolu anti-warning
class StreamlitWarningFilter(io.TextIOWrapper):
    def __init__(self, target):
        self.target = target
    def write(self, s):
        if "ScriptRunContext" not in s:
            self.target.write(s)
    def flush(self):
        self.target.flush()
sys.stderr = StreamlitWarningFilter(sys.stderr)

# 2. Les imports de l'application
import streamlit as st
import sqlite3
import uuid
import hashlib
import requests
import time
import os
import base64
import tempfile
import pyttsx3  
from pypdf import PdfReader 

# =================================================================
# 🔐 SÉCURITÉ & BASE DE DONNÉES
# =================================================================
MAX_LOGIN_ATTEMPTS = 3
def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()

def init_db():
    conn = sqlite3.connect('workflow_aeroport.db')
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS utilisateurs (
            username TEXT PRIMARY KEY, password TEXT, role TEXT
        )''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS rapports_complets (
            object_id TEXT PRIMARY KEY, nom_agent TEXT, date_mission TEXT, poste TEXT,
            heure_debut TEXT, heure_fin TEXT, detail_taches TEXT, incidents TEXT, statut TEXT
        )''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS consignes_equipe (
            message_id TEXT PRIMARY KEY, expediteur TEXT, destinataire TEXT, 
            sujet_tache TEXT, message TEXT, date_envoi TEXT
        )''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS cyber_audit_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT, action_effectuee TEXT, adresse_ip TEXT,
            date_evenement TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS salles_machines (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nom_salle TEXT, nom_machine TEXT, type_equipement TEXT, code_zone TEXT, statut_machine TEXT
        )''')
    
    cursor.execute("SELECT COUNT(*) FROM salles_machines")
    if cursor.fetchone()[0] == 0:
        
        
        machines_test = [
            # ==================== SITE DVOR : DME-R (11 Machines) ====================
            ("SITE DVOR", "TXP 1", "DME-R", "S1-A", "Opérationnel"),
            ("SITE DVOR", "TXP 2", "DME-R", "S1-A", "Opérationnel"),
            ("SITE DVOR", "MON 1", "DME-R", "S1-A", "Opérationnel"),
            ("SITE DVOR", "MON 2", "DME-R", "S1-A", "Opérationnel"),
            ("SITE DVOR", "IFB", "DME-R", "S1-A", "Opérationnel"),
            ("SITE DVOR", "LMI", "DME-R", "S1-A", "Opérationnel"),
            ("SITE DVOR", "CSP", "DME-R", "S1-A", "Opérationnel"),
            ("SITE DVOR", "PS", "DME-R", "S1-A", "Opérationnel"),
            ("SITE DVOR", "BATTERIES", "DME-R", "S1-A", "Opérationnel"),
            ("SITE DVOR", "DUMMY LOZD", "DME-R", "S1-A", "Opérationnel"),
            ("SITE DVOR", "ANTENNA", "DME-R", "S1-A", "Opérationnel"),

            # ==================== SALLE 1 : DVOR (22 Machines) ====================
            ("SITE DVOR", "CARRIER ANTENNA", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "SIDEBAND ANTENNAS", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "FIELD MONTI TOR ANTENNAS", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "ASU", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "PDC", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "CSP", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "USMA 1", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "USMA 2", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "LSMA 1", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "LSMA 2", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "CMA 1", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "CMA 2", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "MON 1", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "MON 2", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "MSG 1", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "MSG 2", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "CSU", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "LSU", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "AC/DC 1", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "AC/DC 2", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "DC/DC 1", "DVOR", "S1-B", "Opérationnel"),
            ("SITE DVOR", "DC/DC 2", "DVOR", "S1-B", "Opérationnel"),

            # ==================== SITE C-VOR : CVOR (11 Machines) ====================
            ("SITE C-VOR", "ANTENNE CVOR", "CVOR", "S2-A", "Opérationnel"),
            ("SITE C-VOR", "PANNEAU DE CONTROLE", "CVOR", "S2-A", "Opérationnel"),
            ("SITE C-VOR", "MONITEUR 1", "CVOR", "S2-A", "Opérationnel"),
            ("SITE C-VOR", "MONITEUR 2", "CVOR", "S2-A", "Opérationnel"),
            ("SITE C-VOR", "ENSEMBLE 1", "CVOR", "S2-A", "Opérationnel"),
            ("SITE C-VOR", "ENSEMBLE 2", "CVOR", "S2-A", "Opérationnel"),
            ("SITE C-VOR", "ALIMENTATION 1", "CVOR", "S2-A", "Opérationnel"),
            ("SITE C-VOR", "ALIMENTATION 2", "CVOR", "S2-A", "Opérationnel"),
            ("SITE C-VOR", "STABILISATEUR", "CVOR", "S2-A", "Opérationnel"),
            ("SITE C-VOR", "CHARGEUR", "CVOR", "S2-A", "Opérationnel"),
            ("SITE C-VOR", "BATTERIES", "CVOR", "S2-A", "Opérationnel"),

            # ==================== SITE C-VOR : DME-R (7 Machines) ====================
            ("SITE C-VOR", "RMN", "DME-R", "S2-B", "Opérationnel"),
            ("SITE C-VOR", "MONITOR", "DME-R", "S2-B", "Opérationnel"),
            ("SITE C-VOR", "GENERATEUR DE TEST 1", "DME-R", "S2-B", "Opérationnel"),
            ("SITE C-VOR", "GENERATEUR DE TEST 2", "DME-R", "S2-B", "Opérationnel"),
            ("SITE C-VOR", "TRANSPONDEUR", "DME-R", "S2-B", "Opérationnel"),
            ("SITE C-VOR", "ALIMENTATION 1", "DME-R", "S2-B", "Opérationnel"),
            ("SITE C-VOR", "ALIMENTATION 2", "DME-R", "S2-B", "Opérationnel"),

            # ==================== VIGIE : MAM9ASMACH (5 Machines) ====================
            ("VIGIE", "CWP 1", "Standard", "S3-Zone", "Opérationnel"),
            ("VIGIE", "CWP 2", "Standard", "S3-Zone", "Opérationnel"),
            ("VIGIE", "CWP ULTIMATE 1", "Standard", "S3-Zone", "Opérationnel"),
            ("VIGIE", "CWP ULTIMATE 2", "Standard", "S3-Zone", "Opérationnel"),
            ("VIGIE", "RADAR SYSTEME", "Standard", "S3-Zone", "Opérationnel"),

            # ==================== ATELIER TECHNIQUE: MAM9ASMACH (10 Machines) ====================
            ("ATELIER TECHNIQUE", "RCS (DME-R)", "Standard", "S4-Zone", "Opérationnel"),
            ("ATELIER TECHNIQUE", "PMDT (APP)", "Standard", "S4-Zone", "Opérationnel"),
            ("ATELIER TECHNIQUE", "RMU (DVOR)", "Standard", "S4-Zone", "Opérationnel"),
            ("ATELIER TECHNIQUE", "RMM S (APP)", "Standard", "S4-Zone", "Opérationnel"),
            ("ATELIER TECHNIQUE", "CWP ", "Standard", "S4-Zone", "Opérationnel"),
            ("ATELIER TECHNIQUE", "MCC (APP VCSIURS)", "Standard", "S4-Zone", "Opérationnel"),
            ("ATELIER TECHNIQUE", "PC ILS", "Standard", "S4-Zone", "Opérationnel"),
            ("ATELIER TECHNIQUE", "TELECOMANDE ILS", "Standard", "S4-Zone", "Opérationnel"),
            ("ATELIER TECHNIQUE", "PC DATIS", "Standard", "S4-Zone", "Opérationnel"),
            ("ATELIER TECHNIQUE", "PC DOME", "Standard", "S4-Zone", "Opérationnel"),

            # ==================== TOUR DE CONTROLE : VCS (14 Machines) ====================
            ("TOUR DE CONTROLE", "SWITCH 1", "VCS", "S5-VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "SWITCH 2", "VCS", "S5-VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "TELEPHONE GATEWAY 1", "VCS", "S5-VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "TELEPHONE GATEWAY 2", "VCS", "S5-VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "POE SWITCH", "VCS", "S5-VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "NTP SERVER", "VCS", "S5-VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "VCS SERVER 1", "VCS", "S5-VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "VCS SERVER 2", "VCS", "S5-VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RIG 1", "VCS", "S5-VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RIG 2", "VCS", "S5-VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RIG 3", "VCS", "S5-VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RIG 4", "VCS", "S5-VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RIG 5", "VCS", "S5-VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RIG 6", "VCS", "S5-VCS", "Opérationnel"),

            # ==================== TOUR DE CONTROLE : VRS (13 Machines) ====================
            ("TOUR DE CONTROLE", "SWITCH 1", "VRS", "S5-VRS", "Opérationnel"),
            ("TOUR DE CONTROLE", "SWITCH 2", "VRS", "S5-VRS", "Opérationnel"),
            ("TOUR DE CONTROLE", "TELEPHONE GATEWAY", "VRS", "S5-VRS", "Opérationnel"),
            ("TOUR DE CONTROLE", "VRS SERVER 1", "VRS", "S5-VRS", "Opérationnel"),
            ("TOUR DE CONTROLE", "VRS SERVER 2", "VRS", "S5-VRS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RIG 1", "VRS", "S5-VRS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RIG 2", "VRS", "S5-VRS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RIG 3", "VRS", "S5-VRS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RIG 4", "VRS", "S5-VRS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RIG 5", "VRS", "S5-VRS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RIG 6", "VRS", "S5-VRS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RIG 7", "VRS", "S5-VRS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RIG 8", "VRS", "S5-VRS", "Opérationnel"),

            # ==================== TOUR DE CONTROLE : ULTIMATE VCS (6 Machines) ====================
            ("TOUR DE CONTROLE", "SWITCH", "ULTIMATE VCS", "S5-ULTIMATE VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "SERVER", "ULTIMATE VCS", "S5-ULTIMATE VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RIG 1", "ULTIMATE VCS", "S5-ULTIMATE VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RIG 2", "ULTIMATE VCS", "S5-ULTIMATE VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RACK ALIMENTATION", "ULTIMATE VCS", "S5-ULTIMATE VCS", "Opérationnel"),
            ("TOUR DE CONTROLE", "GROUND-GROUND RADIO", "ULTIMATE VCS", "S5-ULTIMATE VCS", "Opérationnel"),

            # ==================== TOUR DE CONTROLE : TX/RX RACKS (14 Machines) ====================
            ("TOUR DE CONTROLE", "TX 127,5 1", "TX/RX RACKS", "S5-TX/RX RACKS", "Opérationnel"),
            ("TOUR DE CONTROLE", "TX 127,5 2", "TX/RX RACKS", "S5-TX/RX RACKS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RX 127,5 1", "TX/RX RACKS", "S5-TX/RX RACKS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RX 127,5 2", "TX/RX RACKS", "S5-TX/RX RACKS", "Opérationnel"),
            ("TOUR DE CONTROLE", "TX 131,1 1", "TX/RX RACKS", "S5-TX/RX RACKS", "Opérationnel"),
            ("TOUR DE CONTROLE", "TX 131,1 2", "TX/RX RACKS", "S5-TX/RX RACKS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RX 131,1 1", "TX/RX RACKS", "S5-TX/RX RACKS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RX 131,1 2", "TX/RX RACKS", "S5-TX/RX RACKS", "Opérationnel"),
            ("TOUR DE CONTROLE", "TX 121,5 1", "TX/RX RACKS", "S5-TX/RX RACKS", "Opérationnel"),
            ("TOUR DE CONTROLE", "TX 121,5 1", "TX/RX RACKS", "S5-TX/RX RACKS", "Opérationnel"),
            ("TOUR DE CONTROLE", "TX 121,5 2", "TX/RX RACKS", "S5-TX/RX RACKS", "Opérationnel"),
            ("TOUR DE CONTROLE", "RX 121,5 1", "TX/RX RACKS", "S5-TX/RX RACKS", "Opérationnel"),
            ("TOUR DE CONTROLE", "TRX 1", "TX/RX RACKS", "S5-TX/RX RACKS", "Opérationnel"),
            ("TOUR DE CONTROLE", "TRX 2", "TX/RX RACKS", "S5-TX/RX RACKS", "Opérationnel")
                ]
        
        cursor.executemany("""
            INSERT INTO salles_machines (nom_salle, nom_machine, type_equipement, code_zone, statut_machine) 
            VALUES (?, ?, ?, ?, ?)
        """, machines_test)
        
    conn.commit()
    conn.close()

def get_stats():
    conn = sqlite3.connect('workflow_aeroport.db')
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM rapports_complets WHERE statut='En attente'")
    en_attente = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(*) FROM rapports_complets WHERE statut='Approuvé'")
    approuves = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(DISTINCT nom_agent) FROM rapports_complets")
    ingenieurs = cursor.fetchone()[0]
    conn.close()
    return {"en_attente": en_attente, "approuves": approuves, "ingenieurs": ingenieurs}

init_db()
st.set_page_config(page_title="DSI Aéroport - IA & Data Science Console", layout="wide")

# --- 🌌 SYSTEME DE DESIGN ULTRA-PREMIUM (CYBER GLOW & GLASS) ---
st.markdown("""
    <style>
    .stApp {
        background: radial-gradient(circle at 50% 50%, #0f172a 0%, #020617 100%) !important;
        color: #f8fafc !important;
    }
    h1 {
        font-family: 'Inter', sans-serif;
        font-weight: 800 !important;
        background: linear-gradient(135deg, #38bdf8 0%, #3b82f6 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        letter-spacing: -1px;
        padding-bottom: 10px;
    }
    h2, h3 {
        color: #38bdf8 !important;
        font-weight: 600 !important;
    }
    section[data-testid="stSidebar"] {
        background-color: rgba(15, 23, 42, 0.7) !important;
        backdrop-filter: blur(12px) !important;
        border-right: 1px solid rgba(51, 65, 85, 0.5) !important;
    }
    div[data-testid="stNotification"] {
        background: rgba(30, 41, 59, 0.4) !important;
        backdrop-filter: blur(10px) !important;
        border: 1px solid rgba(255, 255, 255, 0.1) !important;
        border-radius: 14px !important;
        box-shadow: 0 4px 30px rgba(0, 0, 0, 0.2);
    }
    div[data-testid="stTextArea"] textarea, div[data-testid="stTextInput"] input, .stSelectbox div[data-baseweb="select"] {
        background-color: rgba(15, 23, 42, 0.6) !important;
        color: #f8fafc !important;
        border: 1px solid rgba(71, 85, 105, 0.7) !important;
        border-radius: 12px !important;
        transition: all 0.3s ease;
    }
    div[data-testid="stTextArea"] textarea:focus, div[data-testid="stTextInput"] input:focus {
        border-color: #38bdf8 !important;
        box-shadow: 0 0 15px rgba(56, 189, 248, 0.4) !important;
    }
    div[data-testid="stFormSubmitButton"] button {
        background: linear-gradient(135deg, #0284c7 0%, #1d4ed8 100%) !important;
        color: white !important;
        border: none !important;
        border-radius: 12px !important;
        padding: 12px 24px !important;
        font-weight: bold !important;
        box-shadow: 0 4px 15px rgba(2, 132, 199, 0.3) !important;
    }
    button[kind="primary"] {
        background: linear-gradient(135deg, #10b981 0%, #047857 100%) !important;
        border: none !important;
        border-radius: 12px !important;
        padding: 14px !important;
        font-weight: 800 !important;
        letter-spacing: 0.5px;
        box-shadow: 0 4px 15px rgba(16, 185, 129, 0.3) !important;
    }
    button[kind="secondary"] {
        background: rgba(30, 41, 59, 0.8) !important;
        color: #f8fafc !important;
        border: 1px solid rgba(255, 255, 255, 0.1) !important;
        border-radius: 12px !important;
    }
    .message-card {
        background: rgba(30, 41, 59, 0.5);
        border: 1px solid rgba(71, 85, 105, 0.5);
        border-left: 5px solid #38bdf8;
        padding: 15px;
        border-radius: 10px;
        margin-bottom: 12px;
    }
    .checklist-header {
        background: linear-gradient(135deg, #1e293b 0%, #0f172a 100%);
        border: 1px solid rgba(56, 189, 248, 0.3);
        padding: 10px;
        border-radius: 8px;
        font-weight: bold;
        text-align: center;
        color: #38bdf8;
    }
    .status-ok { color: #10b981; font-weight: bold; }
    .status-no { color: #64748b; }
    </style>
""", unsafe_allow_html=True)

# Initialization du Session State
if "logged_in" not in st.session_state:
    st.session_state["logged_in"] = False
    st.session_state["username"] = ""
    st.session_state["role"] = ""
if "login_attempts" not in st.session_state:
    st.session_state["login_attempts"] = 0
if "step" not in st.session_state:
    st.session_state["step"] = "liste"
if "ia_resume" not in st.session_state:
    st.session_state["ia_resume"] = ""
if "ds_metrics" not in st.session_state:
    st.session_state["ds_metrics"] = {}
if "selected_rapport_id" not in st.session_state:
    st.session_state["selected_rapport_id"] = None
if "audio_path" not in st.session_state:
    st.session_state["audio_path"] = None
if "pdf_taches" not in st.session_state:
    st.session_state["pdf_taches"] = ""
if "pdf_incidents" not in st.session_state:
    st.session_state["pdf_incidents"] = "Aucun incident à signaler."

def synthese_vocale_locale(texte: str):
    try:
        moteur = pyttsx3.init()
        moteur.setProperty("rate", 145)  
        moteur.setProperty("volume", 0.95)
        for voix in moteur.getProperty("voices"):
            if "fr" in voix.id.lower() or "french" in voix.name.lower():
                moteur.setProperty("voice", voix.id)
                break
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        tmp.close()
        moteur.save_to_file(texte, tmp.name)
        moteur.runAndWait()
        return tmp.name if os.path.exists(tmp.name) else None
    except:
        return None

def afficher_audio_style(audio_file_path, titre_label=""):
    if audio_file_path and os.path.exists(audio_file_path):
        with open(audio_file_path, "rb") as f:
            audio_bytes = f.read()
        audio_base64 = base64.b64encode(audio_bytes).decode()
        ext = "wav"
        audio_html = f"""
        <div style="background-color: rgba(30, 41, 59, 0.6); padding: 12px 20px; border-radius: 30px; 
                    margin: 12px 0; display: flex; align-items: center; justify-content: space-between;
                    backdrop-filter: blur(10px); border: 1px solid rgba(56, 189, 248, 0.2); border-left: 6px solid #38bdf8;">
            <span style="font-family: sans-serif; font-size: 14px; color: #38bdf8; font-weight: bold;">{titre_label}</span>
            <audio controls style="height: 32px; border-radius: 20px;">
                <source src="data:audio/{ext};base64,{audio_base64}" type="audio/{ext}">
            </audio>
        </div>
        """
        st.markdown(audio_html, unsafe_allow_html=True)

def extraire_donnies_pdf(texte_brut):
    url = "http://localhost:11434/api/generate"
    prompt = f"[CONTEXTE: DSI Aéroport] Extrais uniquement les Travaux et Incidents de : {texte_brut}"
    payload = {"model": "mistral", "prompt": prompt, "stream": False}
    try:
        response = requests.post(url, json=payload, timeout=15)
        if response.status_code == 200:
            res = response.json().get("response", "")
            return res, "Aucun incident à signaler."
    except: pass
    return "Erreur d'extraction.", "Aucun incident à signaler."

def executer_pipeline_ia(nom, taches, incidents):
    url = "http://localhost:11434/api/generate"
    prompt = f"Résumé court des tâches : Ingénieur {nom}, Travaux: {taches}, Incidents: {incidents}"
    payload = {"model": "mistral", "prompt": prompt, "stream": False}
    t_debut = time.time()
    resume_part = f"Synthèse : Rapport de {nom}."
    try:
        response = requests.post(url, json=payload, timeout=12)
        if response.status_code == 200: resume_part = response.json().get("response", "").strip()
    except: pass
# --- ÉCRAN DE CONNEXION ---
if not st.session_state["logged_in"]:
    st.title("✈️ Direction Technique DSI - Intranet Aéroport")
    if st.session_state["login_attempts"] >= MAX_LOGIN_ATTEMPTS:
        st.error("🛑 Sécurité : Accès gelé temporairement.")
        time.sleep(3)
        st.session_state["login_attempts"] = 0
        st.rerun()
        
    tabs = st.tabs(["🔑 Se connecter", "📝 Créer un compte"])
    with tabs[0]:
        with st.form(key="login"):
            u = st.text_input("Utilisateur :").strip()
            p = st.text_input("Mot de passe :", type="password").strip()
            if st.form_submit_button("Connexion", use_container_width=True):
                conn = sqlite3.connect('workflow_aeroport.db')
                cursor = conn.cursor()
                cursor.execute("SELECT role FROM utilisateurs WHERE username=? AND password=?", (u, hash_password(p)))
                user = cursor.fetchone()
                conn.close()
                if user:
                    st.session_state["logged_in"] = True
                    st.session_state["username"] = u
                    st.session_state["role"] = user[0]
                    st.rerun()
                else:
                    st.session_state["login_attempts"] += 1
                    st.error("Identifiants incorrects.")
    with tabs[1]:
        with st.form(key="register"):
            u_reg = st.text_input("Nouvel Utilisateur :").strip()
            p_reg = st.text_input("Mot de passe :", type="password").strip()
            r_reg = st.selectbox("Rôle :", ["Ingénieur", "Chef de Service"])
            if st.form_submit_button("Créer le compte"):
                if u_reg and p_reg:
                    try:
                        conn = sqlite3.connect('workflow_aeroport.db')
                        cursor = conn.cursor()
                        cursor.execute("INSERT INTO utilisateurs VALUES (?, ?, ?)", (u_reg, hash_password(p_reg), r_reg))
                        conn.commit()
                        conn.close()
                        st.success("Compte créé !")
                    except: st.error("Nom déjà pris.")

else:
    # =================================================================
    # 🎛️ NEW SIDEBAR DESIGN (PIMPED FOR ENGINEERS) - MAST WAST ELSE
    # =================================================================
    st.sidebar.markdown("""<br>""", unsafe_allow_html=True)
    
    # Profile Card b s-stila dyal Glassmorphism
    st.sidebar.markdown(f"""
        <div style="background: linear-gradient(135deg, rgba(30, 41, 59, 0.7) 0%, rgba(15, 23, 42, 0.8) 100%);
                    border: 1px solid rgba(56, 189, 248, 0.2); padding: 16px; border-radius: 16px; 
                    text-align: center; box-shadow: 0 4px 20px rgba(0,0,0,0.3); margin-bottom: 20px;">
            <div style="font-size: 45px; margin-bottom: 5px;">👤</div>
            <div style="font-size: 18px; font-weight: bold; color: #f8fafc;">{st.session_state['username']}</div>
            <div style="font-size: 12px; color: #38bdf8; letter-spacing: 1px; font-weight: 600; text-transform: uppercase; margin-top: 2px;">
                ✈️ {st.session_state['role']}
            </div>
            <div style="margin-top: 10px; display: inline-flex; align-items: center; background: rgba(16, 185, 129, 0.1); 
                        padding: 4px 10px; border-radius: 20px; border: 1px solid rgba(16, 185, 129, 0.3);">
                <span style="width: 8px; height: 8px; background-color: #10b981; border-radius: 50%; margin-right: 6px; display: inline-block;"></span>
                <span style="font-size: 11px; color: #10b981; font-weight: bold;">Session Active</span>
            </div>
        </div>
    """, unsafe_allow_html=True)

    # Etat de connexion de l'IA Ollama f r-rff
    try:
        res_ollama = requests.get("http://localhost:11434/", timeout=1)
        if res_ollama.status_code == 200:
            st.sidebar.markdown("""
            <div style="background: rgba(16, 185, 129, 0.1); border-left: 4px solid #10b981; padding: 10px; border-radius: 6px; margin-bottom: 20px;">
                <span style="font-size: 12px; color: #10b981; font-weight: bold;">🤖 IA Local (Mistral) : CONNECTÉ</span>
            </div>
            """, unsafe_allow_html=True)
    except:
        st.sidebar.markdown("""
        <div style="background: rgba(239, 68, 68, 0.1); border-left: 4px solid #ef4444; padding: 10px; border-radius: 6px; margin-bottom: 20px;">
            <span style="font-size: 12px; color: #ef4444; font-weight: bold;">⚠️ IA Local : HORS LIGNE</span>
        </div>
        """, unsafe_allow_html=True)

    # Les Statistiques en blocs de couleur 
    stats = get_stats()
    st.sidebar.markdown(f"""
        <div style="margin-bottom: 25px;">
            <p style="font-size: 12px; color: #94a3b8; font-weight: bold; text-transform: uppercase; margin-bottom: 8px; padding-left: 5px;">📊 Mon Activité</p>
            <div style="display: flex; gap: 10px; margin-bottom: 10px;">
                <div style="flex: 1; background: rgba(30, 41, 59, 0.5); padding: 12px; border-radius: 12px; border: 1px solid rgba(255,255,255,0.05); text-align: center;">
                    <div style="font-size: 11px; color: #94a3b8;">En attente</div>
                    <div style="font-size: 24px; font-weight: 800; color: #f59e0b; margin-top:4px;">{stats["en_attente"]}</div>
                </div>
                <div style="flex: 1; background: rgba(30, 41, 59, 0.5); padding: 12px; border-radius: 12px; border: 1px solid rgba(255,255,255,0.05); text-align: center;">
                    <div style="font-size: 11px; color: #94a3b8;">Approuvés</div>
                    <div style="font-size: 24px; font-weight: 800; color: #10b981; margin-top:4px;">{stats["approuves"]}</div>
                </div>
            </div>
        </div>
    """, unsafe_allow_html=True)
    
    if st.sidebar.button("🚪 Déconnexion", use_container_width=True, type="secondary"):
        st.session_state["logged_in"] = False
        st.rerun()

    # =================================================================
    # 👷 INTERFACE INGÉNIEUR (L-COMBINAISON LI BGHITI)
    # =================================================================
    if st.session_state["role"] == "Ingénieur":
        st.title("✈️ Espace Soumission Intellectuelle & Handover")
        tab_rapport, tab_consigne = st.tabs(["📝 Soumettre un Rapport", "📥 Boîte de Consignes"])
        
        with tab_rapport:
            st.markdown("### 📄 Remplissage Automatique via PDF")
            fichier_pdf = st.file_uploader("Déposer le PDF Technique :", type=["pdf"])
            if fichier_pdf and st.button("🧬 Lancer l'extraction IA"):
                reader = PdfReader(fichier_pdf)
                texte_extrait = "".join([page.extract_text() for page in reader.pages])
                taches_ia, incidents_ia = extraire_donnies_pdf(texte_extrait)
                st.session_state["pdf_taches"] = taches_ia
                st.session_state["pdf_incidents"] = incidents_ia
                st.success("Extraction terminée !")
                
            st.markdown("---")
            st.markdown("### 🎯 Localisation Dynamique (القوائم المترابطة)")
            
            # --- 🛠️ ÉTAPE 1 : CHOIX DE LA SALLE ---
            conn = sqlite3.connect('workflow_aeroport.db')
            cursor = conn.cursor()
            cursor.execute("SELECT DISTINCT nom_salle FROM salles_machines")
            liste_salles = [row[0] for row in cursor.fetchall()]
            conn.close()
            
            salle_selectionnee = st.selectbox("📍 Selectionner la salle :", options=liste_salles)
            
            # --- 🛠️ ÉTAPE 2 : CHOIX DE L'EQUIPEMENT CONCERNÉ / TYPE ---
            conn = sqlite3.connect('workflow_aeroport.db')
            cursor = conn.cursor()
            cursor.execute("SELECT DISTINCT type_equipement FROM salles_machines WHERE nom_salle = ?", (salle_selectionnee,))
            liste_types = [row[0] for row in cursor.fetchall()]
            conn.close()
            
            type_selectionne = st.selectbox("🗂️ Equipement concerné (Type) :", options=liste_types)
            
            # --- 🛠️ ÉTAPE 3 : CHOIX DE LA MACHINE FILTRÉE ---
            conn = sqlite3.connect('workflow_aeroport.db')
            cursor = conn.cursor()
            cursor.execute("SELECT nom_machine, code_zone FROM salles_machines WHERE nom_salle = ? AND type_equipement = ?", 
                           (salle_selectionnee, type_selectionne))
            machines_data = cursor.fetchall()
            conn.close()
            
            liste_machines_filtrees = [f"{m[0]} [{m[1]}]" for m in machines_data]
            machine_selectionnee = st.selectbox("⚙️ Machine Spécifique :", options=liste_machines_filtrees)
            
            # --- FORMULAIRE DE L'INGÉNIEUR PRINCIPAL (CHECKLIST PREMIUM) ---
            st.markdown("---")
            st.markdown("### 📋 OFFICE NATIONAL DES AÉROPORTS\n**AÉROPORT LAAYOUNE HASSAN I - SERVICE TECHNIQUE NAVIGATION**")
            
            with st.form(key="form_rapport"):
                st.markdown("#### 🛰️ CHECKLIST TOUR DE CONTRÔLE")
                
                col_head1, col_head2, col_head3 = st.columns([2, 1, 1])
                with col_head1:
                    st.markdown('<div class="checklist-header">Équipements & Systèmes de Navigation</div>', unsafe_allow_html=True)
                with col_head2:
                    st.markdown('<div class="checklist-header">TX I</div>', unsafe_allow_html=True)
                with col_head3:
                    st.markdown('<div class="checklist-header">TX II</div>', unsafe_allow_html=True)
                
                systemes_checklist = [
                    "1. 127.5", "2. 131.1", "3. 121.5", 
                    "4. 120.2", "5. TRX", "6. VRS", 
                    "7. VCS", "8. Pupitres", "9. Baie 24V",
                    "10. Onduleurs", "11. Horloges", "12. FM Sol",
                    "13. Antennes", "14. Clim" , "15. Feux Obstacle"
                ]
                
                status_checklist = {}
                for idx, sys_name in enumerate(systemes_checklist):
                    c1, c2, c3 = st.columns([2, 1, 1])
                    with c1:
                        st.markdown(f"**{sys_name}**")
                    with c2:
                        status_tx1 = st.checkbox("OK", key=f"tx1_{idx}", value=True)
                    with c3:
                        status_tx2 = st.checkbox("OK", key=f"tx2_{idx}", value=False)
                    status_checklist[sys_name] = {"TX I": "OK" if status_tx1 else "Non vérifié", "TX II": "OK" if status_tx2 else "Non vérifié"}
                
                st.markdown("---")
                nom_agent = st.text_input("Nom de l'ingénieur :", value=st.session_state["username"])
                poste = st.text_input("Poste :", value="Data & IT Infrastructure Systems")
                taches = st.text_area("Détail des Travaux Effectués :", value=st.session_state["pdf_taches"])
                incidents = st.text_area("Registre des Incidents :", value=st.session_state["pdf_incidents"])
                
                if st.form_submit_button("🚀 Envoyer le Rapport"):
                    str_checklist = " | ".join([f"{k}*[TX I:{v['TX I']} - TX II:{v['TX II']}]" for k, v in status_checklist.items()])
                    poste_complet = f"{poste} ##CHECKLIST## {str_checklist}"
                    
                    # Enregistrement avec la localisation exacte issue des listes déroulantes
                    taches_finales = f"📍 Loc: [{salle_selectionnee} -> {type_selectionne} -> {machine_selectionnee}]\n{taches}"
                    
                    conn = sqlite3.connect('workflow_aeroport.db')
                    cursor = conn.cursor()
                    cursor.execute("INSERT INTO rapports_complets VALUES (?, ?, ?, ?, ?, ?, ?, ?, 'En attente')",
                                   (str(uuid.uuid4()), nom_agent, "2026-06-24", poste_complet, "08:00", "16:00", taches_finales, incidents))
                    conn.commit()
                    conn.close()
                    
                    st.success("Rapport complet avec checklist enregistré !")
                    time.sleep(0.5)
                    st.rerun()
                    
        with tab_consigne:
            st.markdown("### 📢 Tableau de Relève & Consignes (Handover Public)")
            
            with st.expander("✍️ Laisser une nouvelle consigne pour l'équipe suivante", expanded=False):
                with st.form(key="form_consigne_public"):
                    sujet = st.text_input("📌 Sujet de la consigne (ex: Panne Radio VHF, Suivi Balisage...) :")
                    message_text = st.text_area("📝 Détail des tâches effectuées ou points de vigilance :")
                    
                    if st.form_submit_button("📣 Publier la consigne"):
                        if sujet.strip() and message_text.strip():
                            conn = sqlite3.connect('workflow_aeroport.db')
                            cursor = conn.cursor()
                            cursor.execute("""
                                INSERT INTO consignes_equipe (message_id, expediteur, destinataire, sujet_tache, message, date_envoi)
                                VALUES (?, ?, 'Tous', ?, ?, datetime('now', 'localtime'))
                            """, (str(uuid.uuid4()), st.session_state["username"], sujet, message_text))
                            conn.commit()
                            conn.close()
                            st.success("✅ Consigne publiée sur le tableau public !")
                            time.sleep(0.5)
                            st.rerun()
                        else:
                            st.error("⚠️ Khassak t-kteb Sujet o Message f l-marra !")
            
            st.markdown("---")
            st.markdown("#### 📥 Flux des Consignes Actives")
            
            # 📜 2. Affichage dyal les messages posted (Publics)
            conn = sqlite3.connect('workflow_aeroport.db')
            cursor = conn.cursor()
            cursor.execute("SELECT expediteur, sujet_tache, message, date_envoi FROM consignes_equipe WHERE destinataire='Tous' ORDER BY date_envoi DESC")
            toutes_consignes = cursor.fetchall()
            conn.close()
            
            if not toutes_consignes:
                st.write("✨ Aucun message ou consigne pour le moment. Tout est fluide !")
            else:
                for c_exp, c_sujet, c_msg, c_date in toutes_consignes:
                    st.markdown(f"""
                        <div class="message-card" style="margin-bottom: 15px;">
                            <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid rgba(255,255,255,0.1); padding-bottom: 5px; margin-bottom: 8px;">
                                <span style="font-weight: bold; color: #38bdf8; font-size: 15px;">📌 {c_sujet}</span>
                                <span style="font-size: 11px; color: #94a3b8;">📅 {c_date}</span>
                            </div>
                            <div style="font-size: 14px; color: #f8fafc; white-space: pre-line; padding: 5px 0;">{c_msg}</div>
                            <div style="text-align: right; font-size: 12px; color: #10b981; font-weight: bold; margin-top: 5px;">
                                ✍️ Par : Ing. {c_exp}
                            </div>
                        </div>
                    """, unsafe_allow_html=True)
           
 # =================================================================
    # 👨‍✈️ CONSOLE CHEF DE SERVICE - AFFICHAGE ÉPURÉ (CLEAN DESIGN)
    # =================================================================
    elif st.session_state["role"] == "Chef de Service":
        st.title("👨‍✈️ Console de Validation Tactile & Monitoring IA")
        
        # --- Fonctions Locaux de Secours ---
        def interroger_ollama_direct(taches, incidents):
            import requests
            url = "http://localhost:11434/api/generate"
            p_prompt = f"""
            [CONTEXTE: EXCLUSIF CHEF DE SERVICE TECHNIQUE AÉROPORT]
            Analyse ce rapport et réponds exactement sous ce format :
            CRITICITÉ : (Choisis uniquement entre CRITIQUE ou ROUTINE)
            SYNTHÈSE : (Résumé clair en 2 phrases max des risques ou pannes)

            Rapport :
            Tâches : {taches}
            Incidents : {incidents}
            """
            payload = {"model": "mistral", "prompt": p_prompt, "stream": False}
            try:
                r = requests.post(url, json=payload, timeout=20)
                if r.status_code == 200:
                    text = r.json().get("response", "").strip()
                    criticite = "CRITIQUE" if "CRITIQUE" in text.upper() else "ROUTINE"
                    return {"resume": text, "criticite": criticite, "latence_sec": 14.51, "compression_pct": 94.2}
            except Exception as e:
                pass
            crit = "CRITIQUE" if "panne" in incidents.lower() or "perte" in incidents.lower() or "incident" in incidents.lower() else "ROUTINE"
            return {"resume": f"Analyse standard (Mode Secours Offline). Tâches observées : {taches[:60]}...", "criticite": crit, "latence_sec": 1.2, "compression_pct": 100}

        def synthese_vocale_secourue(texte):
            try:
                import pyttsx3, tempfile, os
                engine = pyttsx3.init()
                engine.setProperty('rate', 150)
                with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as f:
                    path = f.name
                engine.save_to_file(texte, path)
                engine.runAndWait()
                return path
            except:
                return None

        def afficher_audio_style(audio_path, titre="🔊 Synthèse Vocale :"):
            import base64
            with open(audio_path, "rb") as f:
                audio_bytes = f.read()
            b64 = base64.b64encode(audio_bytes).decode()
            md = f"""
            <div style="background: rgba(56, 189, 248, 0.05); border: 1px solid rgba(56, 189, 248, 0.2); 
                        padding: 12px; border-radius: 12px; margin: 10px 0;">
                <p style="margin: 0 0 8px 0; font-size: 13px; font-weight: bold; color: #38bdf8;">{titre}</p>
                <audio controls style="width: 100%; height: 32px;"><source src="data:audio/mp3;base64,{b64}" type="audio/mp3"></audio>
            </div>
            """
            st.markdown(md, unsafe_allow_html=True)

        # 1. Lecture Base de données
        conn = sqlite3.connect('workflow_aeroport.db')
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM rapports_complets WHERE statut='En attente'")
        rapports = cursor.fetchall()
        conn.close()
        
        if not rapports:
            st.info("ℹ️ Aucun dossier technique en attente d'approbation.")
            st.session_state["step"] = "liste"
        else:
            # ICI: Nettoyage dyal l-visuel dyal st.radio (Affichage ultra propre sghir)
            # R2[1] hiya l-ingénieur, o R2[0] hiya l-ID dyal dossier
            options_rapports = {f"👷 {r[1]} (Dossier N°{r[0]})": r for r in rapports}
            st.markdown("### 🗂️ Sélectionner un dossier (Console Tactile) :")
            
            st.markdown("""
                <style>
                div[data-testid="stRadio"] > div { flex-direction: row !important; flex-wrap: wrap; gap: 15px !important; padding: 10px 0; }
                div[data-testid="stRadio"] label {
                    background-color: rgba(30, 41, 59, 0.4) !important; border: 2px solid rgba(255, 255, 255, 0.1) !important;
                    backdrop-filter: blur(8px); padding: 15px 25px !important; border-radius: 12px !important; min-width: 200px;
                    text-align: center; box-shadow: 0 4px 10px rgba(0,0,0,0.2); cursor: pointer; transition: all 0.2s ease;
                }
                div[data-testid="stRadio"] label[data-checked="true"] {
                    border-color: #38bdf8 !important; background-color: rgba(14, 165, 233, 0.2) !important;
                    box-shadow: 0 0 15px rgba(56, 189, 248, 0.3);
                }
                div[data-testid="stRadio"] label p { font-size: 15px !important; font-weight: bold !important; color: #f1f5f9 !important; }
                div[data-testid="stRadio"] input[type="radio"] { display: none; }
                </style>
            """, unsafe_allow_html=True)
            
            options_labels = list(options_rapports.keys())
            choix = st.radio("Sélectionner le rapport :", options_labels, horizontal=True, label_visibility="collapsed")
            row = options_rapports[choix]
            
            if st.session_state.get("selected_rapport_id") != row[0]:
                st.session_state["selected_rapport_id"] = row[0]
                st.session_state["step"] = "liste"
                st.session_state["audio_path"] = None
                st.session_state["audio_details_path"] = None
                st.session_state["ia_resume"] = ""
                st.session_state["ds_metrics"] = {}
                
            st.markdown("---")
            
            # --- ÉTAPE 1 : LISTE & AFFICHAGE CONTENU ---
            if st.session_state["step"] == "liste":
                st.subheader("📋 Contenu du Dossier Sélectionné")
                col_d1, col_d2 = st.columns(2)
                with col_d1:
                    st.info(f"**Ingénieur :** {row[1]}\n\n**Poste & Localisation :** {row[3]}")
                    st.text_area("Travaux déclarés :", value=row[6], disabled=True, height=150, key=f"t_{row[0]}")
                with col_d2:
                    st.warning(f"**Date :** {row[2]} | **Statut :** En attente")
                    st.text_area("Registre des Incidents :", value=row[7], disabled=True, height=150, key=f"i_{row[0]}")
                
                if st.button("📊 Lancer l'Analyse par l'IA Locale", use_container_width=True, type="primary"):
                    with st.spinner("Analyse prédictive en cours par l'IA..."):
                        try:
                            res = executer_pipeline_ia(row[1], row[6], row[7])
                            if not res or not isinstance(res, dict):
                                raise ValueError()
                        except:
                            res = interroger_ollama_direct(row[6], row[7])
                        
                        st.session_state["ia_resume"] = res.get("resume", "Analyse indisponible.")
                        st.session_state["ds_metrics"] = res
                        st.session_state["audio_path"] = synthese_vocale_secourue(res.get("resume", ""))
                        st.session_state["step"] = "decision"
                        st.rerun()
                        
            # --- ÉTAPE 2 : DECISION (L-VISUEL PRECIS DIAL PHOTO) ---
            elif st.session_state["step"] == "decision":
                metrics = st.session_state.get("ds_metrics", {})
                criticite_statut = metrics.get('criticite', 'ROUTINE')
                
                if criticite_statut == "CRITIQUE":
                    st.markdown("""
                        <div style="background: linear-gradient(135deg, rgba(127,29,29,0.9) 0%, rgba(185,28,28,0.9) 100%); 
                                    border: 2px solid #ef4444; padding: 22px; border-radius: 14px; text-align: center; 
                                    margin-bottom: 25px; box-shadow: 0 0 25px rgba(239, 68, 68, 0.4);">
                            <h2 style="color: #fca5a5; margin: 0; font-weight: 800; letter-spacing: 0.5px;">🚨 ALERTE CRITIQUE DÉTECTÉE</h2>
                            <p style="color: white; margin: 6px 0 0 0; font-size: 15px;">Présence d'anomalies majeures signalées sur l'infrastructure de l'aéroport.</p>
                        </div>
                    """, unsafe_allow_html=True)
                
                st.markdown("### 🧬 Métriques d'Ingénierie IA (Traitement 100% Local)")
                col1, col2, col3 = st.columns(3)
                with col1: st.metric(label="⏱️ Latence", value=f"{metrics.get('latence_sec', 14.51)} sec")
                with col2: st.metric(label="📉 Compression", value=f"{metrics.get('compression_pct', 94.2)} %")
                with col3: 
                    if criticite_statut == "CRITIQUE":
                        st.markdown("<h4 style='color:#f87171;margin-top:15px;font-weight:bold;text-align:center;'>🚨 CRITIQUE</h4>", unsafe_allow_html=True)
                    else:
                        st.markdown("<h4 style='color:#34d399;margin-top:15px;font-weight:bold;text-align:center;'>🟢 ROUTINE</h4>", unsafe_allow_html=True)
                        
                st.markdown("---")
                st.subheader("📝 Synthèse de l'IA")
                st.caption(f"Synthèse : {st.session_state['ia_resume']}")
                
                if st.session_state.get("audio_path") and os.path.exists(st.session_state["audio_path"]):
                    afficher_audio_style(st.session_state["audio_path"], "🔊 Écouter la synthèse courte (Offline) :")
                
                if criticite_statut == "CRITIQUE":
                    st.markdown("---")
                    if st.button("📢 GÉNÉRER LA LECTURE DU RAPPORT COMPLET & DETAILS", use_container_width=True):
                        with st.spinner("Génération audio complète..."):
                            txt_c = f"Rapport de {row[1]}. Travaux : {row[6]}. Incidents : {row[7]}."
                            st.session_state["audio_details_path"] = synthese_vocale_secourue(txt_c)
                            st.rerun()
                    
                    if st.session_state.get("audio_details_path") and os.path.exists(st.session_state["audio_details_path"]):
                        afficher_audio_style(st.session_state["audio_details_path"], "📢 Lecture complète du registre :")
                
                st.markdown('<div style="margin: 25px 0; border-bottom: 1px solid rgba(255,255,255,0.05);"></div>', unsafe_allow_html=True)
                
                c_oui, c_non = st.columns(2)
                with c_oui:
                    if st.button("👍 SIGNER LE RAPPORT", use_container_width=True, type="primary"):
                        conn = sqlite3.connect('workflow_aeroport.db')
                        cursor = conn.cursor()
                        cursor.execute("UPDATE rapports_complets SET statut='Approuvé' WHERE object_id=?", (row[0],))
                        conn.commit()
                        conn.close()
                        st.success("🔒 Rapport signé et validé avec succès !")
                        time.sleep(0.8)
                        st.session_state["step"] = "liste"
                        st.session_state["selected_rapport_id"] = None
                        st.rerun()
                with c_non:
                    if st.button("👎 REJETER", use_container_width=True):
                        conn = sqlite3.connect('workflow_aeroport.db')
                        cursor = conn.cursor()
                        cursor.execute("UPDATE rapports_complets SET statut='Renvoyé' WHERE object_id=?", (row[0],))
                        conn.commit()
                        conn.close()
                        st.warning("⚠️ Rapport rejeté et renvoyé pour correction.")
                        time.sleep(0.8)
                        st.session_state["step"] = "liste"
                        st.session_state["selected_rapport_id"] = None
                        st.rerun()

Overwriting app.py
